In [1]:
CSV_PATH = "/kaggle/input/datasets/tranquanghuy2809/routing/qwen_intent_classification.csv"

MODEL   = "keepitreal/vietnamese-sbert"
SEGMENT = True

SEED      = 42
TEST_SIZE = 0.2
N_FOLDS   = 5

# SetFit hyperparams
BATCH_SIZE     = 16
NUM_EPOCHS     = 1
NUM_ITERATIONS = 20      # số cặp ~ 2 * NUM_ITERATIONS * N_train

VALID = ["tra_cuu", "tinh_toan"]
L2I = {n: i for i, n in enumerate(VALID)}
I2L = {i: n for n, i in L2I.items()}
print("MODEL:", MODEL, "| SEGMENT:", SEGMENT)

MODEL: keepitreal/vietnamese-sbert | SEGMENT: True


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

st_model = SentenceTransformer(MODEL, device="cuda" if torch.cuda.is_available() else "cpu")

def embed(texts):
    return st_model.encode(list(texts), normalize_embeddings=True,
                           batch_size=64, show_progress_bar=False)

clf = LogisticRegression(class_weight="balanced", C=4.0, max_iter=3000)
clf.fit(embed(tr_t), tr_y)
report(te_y, [int(p) for p in clf.predict(embed(te_t))], "Embed + LR (TEST)")

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

Xall = embed(texts)
yall = np.array(labels)
make = lambda: LogisticRegression(class_weight="balanced", C=4.0, max_iter=3000)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
sc = cross_val_score(make(), Xall, yall, cv=skf, scoring="f1_macro")
print(f">>> Macro-F1 {N_FOLDS}-fold: {sc.mean():.4f} ± {sc.std():.4f} | {sc.round(4).tolist()}")

report(yall, cross_val_predict(make(), Xall, yall, cv=skf), f"{N_FOLDS}-fold (gộp)")

>>> Macro-F1 5-fold: 0.8597 ± 0.0173 | [0.8777, 0.864, 0.8435, 0.8359, 0.8777]

===== 5-fold (gộp) =====
              precision    recall  f1-score   support

     tra_cuu     0.9898    0.9592    0.9742       907
   tinh_toan     0.6442    0.8816    0.7444        76

    accuracy                         0.9532       983
   macro avg     0.8170    0.9204    0.8593       983
weighted avg     0.9630    0.9532    0.9565       983

Confusion [hàng=thật, cột=dự đoán] (tra_cuu, tinh_toan):
[[870  37]
 [  9  67]]
Macro-F1: 0.8593


0.859344282692547

In [9]:
oof = cross_val_predict(make(), Xall, yall, cv=skf)
d = df.copy()
d["pred"] = [I2L[int(p)] for p in oof]
err = d[d["pred"] != d["intent"]]
print("Tổng lỗi:", len(err))

print("\n===== tra_cuu BỊ NHẦM thành tinh_toan (false positive) =====")
for q in err[err.intent == "tra_cuu"]["question"].tolist():
    print(" •", q)

print("\n===== tinh_toan BỊ BỎ SÓT thành tra_cuu (false negative) =====")
for q in err[err.intent == "tinh_toan"]["question"].tolist():
    print(" •", q)

Tổng lỗi: 46

===== tra_cuu BỊ NHẦM thành tinh_toan (false positive) =====
 • Theo quy định tại tài liệu Public_009, nếu một học phần có cấu trúc 3(2:1:6) thì số tiết lý thuyết, thực hành và tự học lần lượt là bao nhiêu?
 • Với một mạng kết nối 200 máy tính và có bán kính hoạt động 500m trong một tòa nhà, phân loại mạng chính xác nhất của nó là gì?
 • Trong hệ đếm thập lục phân, ký tự “F” biểu diễn giá trị thập phân nào?
 • Theo TCVN 9395:2012, tiêu chuẩn này áp dụng cho loại cọc nào và từ đường kính bao nhiêu trở lên?
 • Theo TCVN 9379:2012, tính toán kết cấu bê tông và bê tông cốt thép cần được thực hiện theo những trạng thái giới hạn nào và phải đảm bảo điều gì trong suốt thời hạn sử dụng công trình?
 • Theo quy định, cần bao nhiêu phép đo chiều dày tối thiểu trên dầm chịu tải?
 • Theo quy định, mẫu thử nghiệm dầm chịu tải phải chịu tải trọng bằng bao nhiêu phần trăm sức kháng mô men thiết kế?
 • Chiều dày vật liệu bọc bảo vệ dạng bản hoặc tấm được phép sai lệch tối đa bao nhiêu % s

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

S = cosine_similarity(Xall)
np.fill_diagonal(S, -1.0)
nn = S.argmax(1)
pairs = {}
for i, j in enumerate(nn):
    if yall[i] != yall[j] and S[i, j] > 0.8:
        pairs[tuple(sorted((i, j)))] = S[i, j]
pairs = sorted(pairs.items(), key=lambda x: -x[1])

print(f"Cặp gần nhau (cosine>0.8) nhưng KHÁC nhãn: {len(pairs)}")
for (i, j), s in pairs[:25]:
    print(f"\nsim={s:.3f}")
    print(f"  [{df['intent'].iloc[i]:9s}] {df['question'].iloc[i]}")
    print(f"  [{df['intent'].iloc[j]:9s}] {df['question'].iloc[j]}")

Cặp gần nhau (cosine>0.8) nhưng KHÁC nhãn: 2

sim=0.997
  [tra_cuu  ] Theo TCVN 9379:2012, tính toán kết cấu bê tông và bê tông cốt thép cần được thực hiện theo những trạng thái giới hạn nào và phải đảm bảo điều gì trong suốt thời hạn sử dụng công trình?
  [tinh_toan] Theo TCVN 9379:2012, tính toán kết cấu bê tông và bê tông cốt thép cần được tiến hành theo những trạng thái giới hạn nào và phải đảm bảo điều gì trong suốt thời hạn sử dụng công trình?

sim=0.805
  [tra_cuu  ] Muốn một vật rắn ở trạng thái cân bằng cần điều kiện gì về lực và điều kiện gì về mômen lực?
  [tinh_toan] Một vật rắn ở trạng thái cân bằng tĩnh phải thỏa mãn điều kiện nào, và trong cân bằng quay thì gia tốc góc của vật có giá trị bao nhiêu?


In [11]:
import random
from collections import defaultdict
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, losses,
                                   SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments)
from sklearn.linear_model import LogisticRegression

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def make_pairs(texts, labels, num_iterations=NUM_ITERATIONS, seed=SEED):
    """Mỗi anchor sinh 1 cặp dương (cùng intent) + 1 cặp âm (khác intent)."""
    rng = random.Random(seed)
    by = defaultdict(list)
    for t, l in zip(texts, labels):
        by[l].append(t)
    classes = list(by)
    s1, s2, lab = [], [], []
    for _ in range(num_iterations):
        for t, l in zip(texts, labels):
            s1.append(t); s2.append(rng.choice(by[l])); lab.append(1.0)              # dương
            other = rng.choice([c for c in classes if c != l])
            s1.append(t); s2.append(rng.choice(by[other])); lab.append(0.0)          # âm
    return Dataset.from_dict({"sentence1": s1, "sentence2": s2, "label": lab})

def setfit_finetune(tr_t, tr_y, te_t, te_y, tag="TEST"):
    model = SentenceTransformer(MODEL, device=DEVICE)
    pairs = make_pairs(tr_t, tr_y)
    print(f"[pairs] {len(pairs)} cặp contrastive")
    trainer = SentenceTransformerTrainer(
        model=model,
        args=SentenceTransformerTrainingArguments(
            output_dir="/tmp/st_ft", num_train_epochs=1,
            per_device_train_batch_size=16, learning_rate=2e-5,
            warmup_ratio=0.1, fp16=True, report_to="none", logging_steps=200,
        ),
        train_dataset=pairs,
        loss=losses.CosineSimilarityLoss(model),   # kéo cùng-intent gần, đẩy khác-intent xa
    )
    trainer.train()

    enc = lambda X: model.encode(list(X), normalize_embeddings=True, batch_size=64)
    clf = LogisticRegression(class_weight="balanced", C=4.0, max_iter=3000)
    clf.fit(enc(tr_t), tr_y)
    return model, clf, report(te_y, [int(p) for p in clf.predict(enc(te_t))], tag)

model_ft, clf_ft, macro_ft = setfit_finetune(tr_t, tr_y, te_t, te_y, "SetFit fine-tune (TEST)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


[pairs] 31440 cặp contrastive


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
200,0.080894
400,0.003525
600,0.001606
800,0.001137


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


===== SetFit fine-tune (TEST) =====
              precision    recall  f1-score   support

     tra_cuu     0.9945    1.0000    0.9973       182
   tinh_toan     1.0000    0.9333    0.9655        15

    accuracy                         0.9949       197
   macro avg     0.9973    0.9667    0.9814       197
weighted avg     0.9950    0.9949    0.9948       197

Confusion [hàng=thật, cột=dự đoán] (tra_cuu, tinh_toan):
[[182   0]
 [  1  14]]
Macro-F1: 0.9814


In [12]:
X = np.array(texts, dtype=object); y = np.array(labels)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

scores = []
for k, (tr, te) in enumerate(skf.split(X, y), 1):
    print(f"\n########## FOLD {k}/{N_FOLDS} ##########")
    _, _, m = setfit_finetune(X[tr].tolist(), y[tr].tolist(),
                              X[te].tolist(), y[te].tolist(), tag=f"FOLD {k}")
    scores.append(m)

scores = np.array(scores)
print(f"\n>>> SetFit fine-tune Macro-F1 {N_FOLDS}-fold: "
      f"{scores.mean():.4f} ± {scores.std():.4f} | {scores.round(4).tolist()}")


########## FOLD 1/5 ##########


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


[pairs] 31440 cặp contrastive


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

...

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
200,0.077726
400,0.002002
600,0.000138
800,0.000089


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


===== FOLD 5 =====
              precision    recall  f1-score   support

     tra_cuu     0.9944    0.9890    0.9917       181
   tinh_toan     0.8750    0.9333    0.9032        15

    accuracy                         0.9847       196
   macro avg     0.9347    0.9611    0.9475       196
weighted avg     0.9853    0.9847    0.9849       196

Confusion [hàng=thật, cột=dự đoán] (tra_cuu, tinh_toan):
[[179   2]
 [  1  14]]
Macro-F1: 0.9475

>>> SetFit fine-tune Macro-F1 5-fold: 0.9473 ± 0.0284 | [1.0, 0.9231, 0.9216, 0.9441, 0.9475]


In [13]:
import joblib, json, os

OUT = "/kaggle/working/intent_setfit"; os.makedirs(OUT, exist_ok=True)

final_model = SentenceTransformer(MODEL, device=DEVICE)
SentenceTransformerTrainer(
    model=final_model,
    args=SentenceTransformerTrainingArguments(
        output_dir="/tmp/st_final", num_train_epochs=1,
        per_device_train_batch_size=16, learning_rate=2e-5,
        warmup_ratio=0.1, fp16=True, report_to="none", logging_steps=200),
    train_dataset=make_pairs(texts, labels),
    loss=losses.CosineSimilarityLoss(final_model),
).train()

enc = lambda X: final_model.encode(list(X), normalize_embeddings=True, batch_size=64)
final_head = LogisticRegression(class_weight="balanced", C=4.0, max_iter=3000)
final_head.fit(enc(texts), labels)

final_model.save(os.path.join(OUT, "st_model"))
joblib.dump(final_head, os.path.join(OUT, "head.joblib"))
json.dump({"labels": VALID, "segment": SEGMENT, "base_model": MODEL},
          open(os.path.join(OUT, "config.json"), "w"), ensure_ascii=False, indent=2)
print("Đã lưu:", OUT)

# Hàm dự đoán + test nhanh 2 ca tiêu biểu (1 tra_cuu có số, 1 tinh_toan thật)
def predict_intent(questions):
    qs = segment(questions) if SEGMENT else list(questions)
    return [I2L[int(i)] for i in final_head.predict(enc(qs))]

print(predict_intent([
    "Hằng số điện môi trong chân không có giá trị bao nhiêu?",     # mong: tra_cuu
    "Có 500 mẫu, 25 bị từ chối, tỷ lệ từ chối là bao nhiêu?",      # mong: tinh_toan
]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
200,0.081364
400,0.002782
600,0.002054
800,0.001371
1000,0.001525
1200,0.001535


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu: /kaggle/working/intent_setfit
['tra_cuu', 'tinh_toan']


In [14]:
GEN_PATH = "/kaggle/input/datasets/tranquanghuy2809/routing/question_gen.csv"   
gen = pd.read_csv(GEN_PATH)
gen["question_gen"] = gen["question_gen"].astype(str).str.strip()
gen["intent"]       = gen["intent"].astype(str).str.strip()
gen = gen[gen["intent"].isin(VALID)].reset_index(drop=True)

pred_gen  = predict_intent(gen["question_gen"].tolist())
pred_orig = predict_intent(gen["question"].tolist())

# (a) Độ bền: model có giữ nguyên nhãn khi câu bị viết lại không?
agree = np.mean([a == b for a, b in zip(pred_orig, pred_gen)])
print(f"Giữ nguyên nhãn khi paraphrase: {agree:.4f}")

# (b) So với gold
report([L2I[x] for x in gen['intent']], [L2I[x] for x in pred_gen],
       "PARAPHRASE (rò rỉ – chỉ tham khảo)")

Giữ nguyên nhãn khi paraphrase: 0.9720

===== PARAPHRASE (rò rỉ – chỉ tham khảo) =====
              precision    recall  f1-score   support

     tra_cuu     0.9773    0.9946    0.9859       737
   tinh_toan     0.8889    0.6531    0.7529        49

    accuracy                         0.9733       786
   macro avg     0.9331    0.8238    0.8694       786
weighted avg     0.9718    0.9733    0.9714       786

Confusion [hàng=thật, cột=dự đoán] (tra_cuu, tinh_toan):
[[733   4]
 [ 17  32]]
Macro-F1: 0.8694


0.869409391194272

In [15]:
gen2 = gen.copy()
gen2["pred_orig"] = pred_orig
gen2["pred_gen"]  = pred_gen
flip = gen2[(gen2.intent == "tinh_toan") & (gen2.pred_gen == "tra_cuu")]
print("tinh_toan -> tra_cuu sau khi paraphrase:", len(flip))
for _, r in flip.iterrows():
    print(f"\n[gốc → {r.pred_orig}] {r['question']}")
    print(f"[gen → {r.pred_gen}] {r['question_gen']}")

tinh_toan -> tra_cuu sau khi paraphrase: 17

[gốc → tinh_toan] Dựa vào thông tin trong tài liệu Public_001, hãy tính số bài báo trung bình mỗi lớp trong nghiên cứu được đề cập.
[gen → tra_cuu] Dựa vào thông tin trong tài liệu Public_001, hãy cho biết tổng số bài báo được phân loại trong nghiên cứu này.

[gốc → tinh_toan] Theo TCVN 9379:2012, tính toán kết cấu bê tông và bê tông cốt thép cần được tiến hành theo những trạng thái giới hạn nào và phải đảm bảo điều gì trong suốt thời hạn sử dụng công trình?
[gen → tra_cuu] Theo TCVN 9379:2012, trong suốt thời gian sử dụng công trình, tính toán kết cấu bê tông và bê tông cốt thép cần đảm bảo điều gì?

[gốc → tinh_toan] Sử dụng công thức quy đổi RGB sang ảnh xám (p = 0.299 * r + 0.587 * g + 0.114 * b, làm tròn tới hàng đơn vị) được đề cập trong tài liệu Public_043, hãy xác định giá trị pixel ảnh xám ứng với pixel ảnh RGB (100, 10, 1).
[gen → tra_cuu] Khi chuyển đổi từ ảnh màu sang ảnh xám, công thức nào được sử dụng để tính giá trị pixel ảnh 

In [5]:
!pip install -q pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 72.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.2 MB/s eta 0:00:00


In [6]:
# ===== bkai vietnamese-bi-encoder — FULL self-contained 5-fold CV =====
import random, numpy as np, pandas as pd, torch
from collections import defaultdict
from datasets import Dataset
from pyvi import ViTokenizer
from sentence_transformers import (SentenceTransformer, losses,
                                   SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# --- config ---
CSV_PATH  = "/kaggle/input/datasets/tranquanghuy2809/routing/qwen_intent_classification.csv"
BKAI_PATH = "/kaggle/input/datasets/tranquanghuy2809/data-embedding/vietnamese-bi-encoder"
SEED, N_FOLDS, NUM_ITERATIONS, SEGMENT = 42, 5, 20, True
VALID = ["tra_cuu", "tinh_toan"]; L2I = {n: i for i, n in enumerate(VALID)}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Device:", DEVICE)

# --- data ---
df = pd.read_csv(CSV_PATH)
df["question"] = df["question"].astype(str).str.strip()
df["intent"]   = df["intent"].astype(str).str.strip()
df = (df[df["intent"].isin(VALID) & (df["question"].str.len() > 0)]
      .drop_duplicates("question").reset_index(drop=True))
texts  = [ViTokenizer.tokenize(t) for t in df["question"]] if SEGMENT else df["question"].tolist()
labels = [L2I[x] for x in df["intent"]]
print(f"[data] {len(df)} mẫu | {df['intent'].value_counts().to_dict()}")

# --- helpers ---
def report(yt, yp, tag=""):
    print(f"\n===== {tag} =====")
    print(classification_report(yt, yp, labels=[0, 1], target_names=VALID, digits=4, zero_division=0))
    print(confusion_matrix(yt, yp, labels=[0, 1]))
    m = f1_score(yt, yp, average="macro"); print(f"Macro-F1: {m:.4f}"); return m

def make_pairs(txt, lab, num_iterations=NUM_ITERATIONS, seed=SEED):
    rng = random.Random(seed); by = defaultdict(list)
    for t, l in zip(txt, lab): by[l].append(t)
    cls = list(by); s1, s2, y = [], [], []
    for _ in range(num_iterations):
        for t, l in zip(txt, lab):
            s1.append(t); s2.append(rng.choice(by[l])); y.append(1.0)
            o = rng.choice([c for c in cls if c != l])
            s1.append(t); s2.append(rng.choice(by[o])); y.append(0.0)
    return Dataset.from_dict({"sentence1": s1, "sentence2": s2, "label": y})

def finetune_eval(tr_t, tr_y, te_t, te_y, model_path, tag="TEST"):
    model = SentenceTransformer(model_path, device=DEVICE)
    SentenceTransformerTrainer(
        model=model,
        args=SentenceTransformerTrainingArguments(
            output_dir="/tmp/bkai_ft", num_train_epochs=1, per_device_train_batch_size=16,
            learning_rate=2e-5, warmup_ratio=0.1, fp16=True, report_to="none", logging_steps=200),
        train_dataset=make_pairs(tr_t, tr_y),
        loss=losses.CosineSimilarityLoss(model),
    ).train()
    enc  = lambda X: model.encode(list(X), normalize_embeddings=True, batch_size=64)
    head = LogisticRegression(class_weight="balanced", C=4.0, max_iter=3000)
    head.fit(enc(tr_t), tr_y)
    return report(te_y, [int(p) for p in head.predict(enc(te_t))], tag)

# --- 5-fold CV ---
X = np.array(texts, dtype=object); yy = np.array(labels)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
scores = []
for k, (tr, te) in enumerate(skf.split(X, yy), 1):
    print(f"\n########## bkai FOLD {k}/{N_FOLDS} ##########")
    scores.append(finetune_eval(X[tr].tolist(), yy[tr].tolist(),
                                X[te].tolist(), yy[te].tolist(), BKAI_PATH, tag=f"FOLD {k}"))
scores = np.array(scores)
print(f"\n>>> bkai  Macro-F1 {N_FOLDS}-fold: {scores.mean():.4f} ± {scores.std():.4f} | {scores.round(4).tolist()}")
print(">>> keepitreal (tham chiếu):     0.9473 ± 0.0284")

Device: cuda
[data] 983 mẫu | {'tra_cuu': 907, 'tinh_toan': 76}

########## bkai FOLD 1/5 ##########


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
200,0.096533
400,0.001906
600,0.000874
800,0.000082


...

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
200,0.090775
400,0.000336
600,0.000088
800,0.000070


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


===== FOLD 5 =====
              precision    recall  f1-score   support

     tra_cuu     0.9944    0.9834    0.9889       181
   tinh_toan     0.8235    0.9333    0.8750        15

    accuracy                         0.9796       196
   macro avg     0.9090    0.9584    0.9319       196
weighted avg     0.9813    0.9796    0.9802       196

[[178   3]
 [  1  14]]
Macro-F1: 0.9319

>>> bkai  Macro-F1 5-fold: 0.9496 ± 0.0170 | [0.9814, 0.9404, 0.9504, 0.9441, 0.9319]
>>> keepitreal (tham chiếu):     0.9473 ± 0.0284


In [13]:
import os, time, numpy as np, torch
from pyvi import ViTokenizer
from sentence_transformers import SentenceTransformer, models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
sync = (lambda: torch.cuda.synchronize()) if DEVICE=="cuda" else (lambda: None)
seg  = lambda L: [ViTokenizer.tokenize(t) for t in L]

MODEL_DIR = "/kaggle/input/datasets/tranquanghuy2809/routing/routing_finetune/st_model"
print("Trong st_model:", os.listdir(MODEL_DIR))

try:                                              # thử load thẳng
    model = SentenceTransformer(MODEL_DIR, device=DEVICE)
    print("Load thẳng OK")
except Exception as e:                            # nếu lỗi Pooling -> dựng lại
    print("Load thẳng lỗi (", type(e).__name__, ") -> dựng lại")
    tr   = models.Transformer(MODEL_DIR)
    pool = models.Pooling(tr.get_word_embedding_dimension(), pooling_mode="mean")
    model = SentenceTransformer(modules=[tr, pool], device=DEVICE)
print("dim =", model.get_sentence_embedding_dimension())

def benchmark(model, name, n=100, batch=64):
    q = ["Có 500 mẫu, 25 bị từ chối, tỷ lệ từ chối là bao nhiêu?"]
    for _ in range(10): model.encode(seg(q), normalize_embeddings=True)
    sync()
    seg_t, enc_t = [], []
    for _ in range(n):
        t0=time.perf_counter(); s=seg(q); t1=time.perf_counter()
        model.encode(s, normalize_embeddings=True); sync(); t2=time.perf_counter()
        seg_t.append((t1-t0)*1000); enc_t.append((t2-t1)*1000)
    big=seg(q*batch); sync()
    t0=time.perf_counter(); model.encode(big, normalize_embeddings=True, batch_size=batch); sync()
    thr=(time.perf_counter()-t0)/batch*1000
    print(f"\n### {name} ({DEVICE}) ###")
    print(f"  pyvi tách từ       : {np.median(seg_t):5.2f} ms/câu")
    print(f"  encode 1 câu       : {np.median(enc_t):5.2f} ms/câu (p95 {np.percentile(enc_t,95):.2f})")
    print(f"  TỔNG 1 câu         : {np.median(seg_t)+np.median(enc_t):5.2f} ms")
    print(f"  encode batch={batch}  : {thr:5.2f} ms/câu (bulk)")

benchmark(model, "sbert (fine-tuned)")

Trong st_model: ['config.json', 'bpe.codes', 'tokenizer_config.json', 'sentence_bert_config.json', 'config_sentence_transformers.json', 'model.safetensors', 'modules.json', 'vocab.txt', 'added_tokens.json']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Load thẳng lỗi ( TypeError ) -> dựng lại


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_58/1778755055.py:18: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pool = models.Pooling(tr.get_word_embedding_dimension(), pooling_mode="mean")
/tmp/ipykernel_58/1778755055.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("dim =", model.get_sentence_embedding_dimension())


dim = 768

### sbert (fine-tuned) (cuda) ###
  pyvi tách từ       :  0.28 ms/câu
  encode 1 câu       : 11.00 ms/câu (p95 11.62)
  TỔNG 1 câu         : 11.27 ms
  encode batch=64  :  0.84 ms/câu (bulk)


In [14]:
import joblib
head = joblib.load("/kaggle/input/datasets/tranquanghuy2809/routing/routing_finetune/head.joblib")
I2L = {0: "tra_cuu", 1: "tinh_toan"}

def pred(qs):
    e = model.encode(seg(qs), normalize_embeddings=True)
    return [I2L[int(i)] for i in head.predict(e)]

print(pred([
    "Hằng số điện môi trong chân không có giá trị bao nhiêu?",   # mong: tra_cuu
    "Có 500 mẫu, 25 bị từ chối, tỷ lệ từ chối là bao nhiêu?",    # mong: tinh_toan
]))

# Nếu hai câu trên ra đúng -> mean pooling khớp -> lưu bản tương thích để sau load thẳng
model.save("/kaggle/working/routing_finetune_v2")
print("Đã lưu: /kaggle/working/routing_finetune_v2 (nhớ commit/đưa thành Dataset để giữ)")

['tra_cuu', 'tinh_toan']


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu: /kaggle/working/routing_finetune_v2 (nhớ commit/đưa thành Dataset để giữ)


In [15]:
import shutil, json, os
SRC = "/kaggle/input/datasets/tranquanghuy2809/routing/routing_finetune"
DST = "/kaggle/working/routing_finetune_v2"

shutil.copy(os.path.join(SRC, "head.joblib"), os.path.join(DST, "head.joblib"))
json.dump({"labels": ["tra_cuu", "tinh_toan"], "segment": True,
           "pooling": "mean", "normalize_embeddings": True},
          open(os.path.join(DST, "intent_config.json"), "w"), ensure_ascii=False, indent=2)
print("v2 giờ có:", os.listdir(DST))

# ---- Code inference chuẩn để tái dùng sau này ----
from sentence_transformers import SentenceTransformer
from pyvi import ViTokenizer
import joblib

enc  = SentenceTransformer(DST)                 # load thẳng, không lỗi
head = joblib.load(os.path.join(DST, "head.joblib"))
I2L  = {0: "tra_cuu", 1: "tinh_toan"}

def route(qs):
    seg = [ViTokenizer.tokenize(q) for q in qs]
    emb = enc.encode(seg, normalize_embeddings=True)
    return [I2L[int(i)] for i in head.predict(emb)]

print(route(["Tính trung bình cộng của 4 và 6?", "Định nghĩa IoT là gì?"]))

v2 giờ có: ['README.md', 'sentence_bert_config.json', 'model.safetensors', 'config_sentence_transformers.json', 'added_tokens.json', 'tokenizer_config.json', 'bpe.codes', 'head.joblib', 'vocab.txt', 'intent_config.json', 'modules.json', 'config.json', '1_Pooling']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

['tinh_toan', 'tra_cuu']
